# Testing for Classification part

In [3]:
from keras.models import load_model
import cv2
import numpy as np

# Load the test set
X_test = np.load('X_test.npy')
y_test = np.load('y_test.npy')

# Load the entire model including architecture and weights
model = load_model('my_model.keras')

# Define the category dictionary
category_dict = {0: 'Biological', 1: 'Cardboard', 2: 'Glass', 3: 'Metal', 4: 'Paper', 5: 'Plastic'}

# Define the input shape expected by the model
input_shape = (100, 100, 3)

# Evaluate the model on the test set
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

# Initialize variables for summary
correct_predictions = 0

# Function to predict and display results
def predict_and_display(img, true_label, model, category_dict, input_shape):
    global correct_predictions
    test_img = cv2.resize(img, (input_shape[1], input_shape[0]))
    test_img_rgb = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)
    test_img_rgb = test_img_rgb / 255.0
    test_img_rgb = np.expand_dims(test_img_rgb, axis=0)
    
    results = model.predict(test_img_rgb)
    
    label = np.argmax(results, axis=1)[0]
    acc = int(np.max(results, axis=1)[0] * 100)
    predicted_category = category_dict[label]
    true_category = category_dict[true_label]

    if acc < 50:
        predicted_category = 'NONE'
    else:
        if label == true_label:
            correct_predictions += 1

    font_scale = 0.3
    thickness = 1
    text_margin = 2

    # Display true label
    cv2.putText(img, f"True: {true_category}", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), thickness)
    # Display predicted label below the true label
    cv2.putText(img, f"Pred: {predicted_category}", (20, 70), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), thickness)
    # Display accuracy below the predicted label
    cv2.putText(img, str(acc) + '%', (20, 110 + 2 * text_margin), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), thickness)

    height, width, _ = img.shape
    aspect_ratio = width / height
    display_img = cv2.resize(img, (500, int(500 / aspect_ratio)))
    cv2.imshow('LIVE', display_img)
    
    return true_category, predicted_category

# Display predictions on test images
for i in range(len(X_test)):
    img = (X_test[i] * 255).astype(np.uint8)  # Convert back to original scale
    true_label = np.argmax(y_test[i])  # Get the true label from one-hot encoding
    true_category, predicted_category = predict_and_display(img, true_label, model, category_dict, input_shape)
    
    print(f"Sample {i+1}: True Label: {true_category}, Predicted Label: {predicted_category}")
    
    k = cv2.waitKey(0)
    if k == 27:  # ESC key
        cv2.destroyAllWindows()
        break  # Break the loop if ESC key is pressed

# Print summary
total_samples = len(X_test)
accuracy = (correct_predictions / total_samples) * 100

print(f"Total samples: {total_samples}")
print(f"Correct predictions: {correct_predictions}")
print(f"Overall accuracy: {accuracy:.2f}%")




38/38 [==============================] - 12s 44ms/step - loss: 0.1490 - accuracy: 0.8642
Test Loss: 0.14897118508815765
Test Accuracy: 0.8641666769981384
1/1 [==============================] - 0s 411ms/step
Sample 1: True Label: Plastic, Predicted Label: Plastic
1/1 [==============================] - 0s 152ms/step
Sample 2: True Label: Paper, Predicted Label: NONE
1/1 [==============================] - 0s 39ms/step
Sample 3: True Label: Metal, Predicted Label: Metal
1/1 [==============================] - 0s 36ms/step
Sample 4: True Label: Glass, Predicted Label: Glass
1/1 [==============================] - 0s 30ms/step
Sample 5: True Label: Plastic, Predicted Label: Plastic
1/1 [==============================] - 0s 32ms/step
Sample 6: True Label: Cardboard, Predicted Label: Glass
1/1 [==============================] - 0s 51ms/step
Sample 7: True Label: Glass, Predicted Label: Plastic
1/1 [==============================] - 0s 35ms/step
Sample 8: True Label: Glass, Predicted Label: Pla

1/1 [==============================] - 0s 33ms/step
Sample 148: True Label: Cardboard, Predicted Label: Metal
1/1 [==============================] - 0s 33ms/step
Sample 149: True Label: Glass, Predicted Label: Glass
1/1 [==============================] - 0s 32ms/step
Sample 150: True Label: Metal, Predicted Label: Metal
1/1 [==============================] - 0s 35ms/step
Sample 151: True Label: Metal, Predicted Label: NONE
1/1 [==============================] - 0s 36ms/step
Sample 152: True Label: Biological, Predicted Label: Glass
1/1 [==============================] - 0s 39ms/step
Sample 153: True Label: Cardboard, Predicted Label: Plastic
1/1 [==============================] - 0s 35ms/step
Sample 154: True Label: Biological, Predicted Label: Metal
1/1 [==============================] - 0s 38ms/step
Sample 155: True Label: Cardboard, Predicted Label: Metal
1/1 [==============================] - 0s 35ms/step
Sample 156: True Label: Glass, Predicted Label: Plastic
1/1 [================

In [ ]:
#code with confusion matrix


from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import cv2
from keras.models import load_model
import numpy as np


# Load the test set
X_test = np.load('X_test.npy')
y_test = np.load('y_test.npy')

# Load the entire model including architecture and weights
model = load_model('my_model.keras')

# Define the category dictionary
category_dict = {0: 'Biological', 1: 'Cardboard', 2: 'Glass', 3: 'Metal', 4: 'Paper', 5: 'Plastic'}

# Define the input shape expected by the model
input_shape = (100, 100, 3)

# Initialize variables
correct_predictions = 0
y_true = []
y_pred = []

# Define the function to predict and display the result
def predict_and_display(img, true_label, model, category_dict, input_shape):
    global correct_predictions, y_true, y_pred
    # Resize and preprocess the image
    test_img = cv2.resize(img, (input_shape[1], input_shape[0]))
    test_img_rgb = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)
    test_img_rgb = test_img_rgb / 255.0
    test_img_rgb = np.expand_dims(test_img_rgb, axis=0)
    
    # Predict using the model
    results = model.predict(test_img_rgb)
    label = np.argmax(results, axis=1)[0]
    acc = int(np.max(results, axis=1)[0] * 100)
    predicted_category = category_dict[label]
    true_category = category_dict[true_label]
    
    # Collect true and predicted labels for the confusion matrix
    y_true.append(true_label)
    y_pred.append(label)

    # Determine if the prediction is correct
    if acc < 50:
        predicted_category = 'NONE'
    else:
        if label == true_label:
            correct_predictions += 1

    # Add text to the displayed image
    font_scale = 0.3
    thickness = 1
    text_margin = 2

    cv2.putText(img, f"True: {true_category}", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), thickness)
    cv2.putText(img, f"Pred: {predicted_category}", (20, 70), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), thickness)
    cv2.putText(img, str(acc) + '%', (20, 110 + 2 * text_margin), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), thickness)

    # Display the image
    height, width, _ = img.shape
    aspect_ratio = width / height
    display_img = cv2.resize(img, (500, int(500 / aspect_ratio)))
    cv2.imshow('LIVE', display_img)
    
    return true_category, predicted_category

# Display predictions on test images
for i in range(len(X_test)):
    img = (X_test[i] * 255).astype(np.uint8)  # Convert back to original scale
    true_label = np.argmax(y_test[i])  # Get the true label from one-hot encoding
    true_category, predicted_category = predict_and_display(img, true_label, model, category_dict, input_shape)
    
    print(f"Sample {i+1}: True Label: {true_category}, Predicted Label: {predicted_category}")
    
    k = cv2.waitKey(0)
    if k == 27:  # ESC key
        cv2.destroyAllWindows()
        break  # Break the loop if ESC key is pressed

# Print summary
total_samples = len(X_test)
accuracy = (correct_predictions / total_samples) * 100

print(f"Total samples: {total_samples}")
print(f"Correct predictions: {correct_predictions}")
print(f"Overall accuracy: {accuracy:.2f}%")

# Generate and display the confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=list(category_dict.values()))
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()


# Testing for Detection part

In [14]:
!pip install ultralytics tensorflow opencv-python-headless numpy

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'E:\\Anaconda3\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.




  Obtaining dependency information for ultralytics from https://files.pythonhosted.org/packages/13/7d/cc6718e94444fd882ce3d7f11137c11ee2198de1e6df87ae69c2c18dd360/ultralytics-8.3.59-py3-none-any.whl.metadata
  Obtaining dependency information for opencv-python-headless from https://files.pythonhosted.org/packages/26/d0/22f68eb23eea053a31655960f133c0be9726c6a881547e6e9e7e2a946c4f/opencv_python_headless-4.10.0.84-cp37-abi3-win_amd64.whl.metadata
  Obtaining dependency information for torchvision>=0.9.0 from https://files.pythonhosted.org/packages/69/55/ce836703ff77bb21582c3098d5311f8ddde7eadc7eab04be9561961f4725/torchvision-0.20.1-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for ultralytics-thop>=2.0.0 from https://files.pythonhosted.org/packages/4a/87/bfd5285f27c23eeec0f609e814a06fc6c7389f37d8703d98e646a9c5fdc3/ultralytics_thop-2.0.13-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/906.8 kB ? eta -:--:--
   ----------------------------

In [16]:
!pip install ultralytics


  Obtaining dependency information for ultralytics from https://files.pythonhosted.org/packages/13/7d/cc6718e94444fd882ce3d7f11137c11ee2198de1e6df87ae69c2c18dd360/ultralytics-8.3.59-py3-none-any.whl.metadata
  Using cached ultralytics-8.3.59-py3-none-any.whl.metadata (35 kB)
  Obtaining dependency information for torchvision>=0.9.0 from https://files.pythonhosted.org/packages/69/55/ce836703ff77bb21582c3098d5311f8ddde7eadc7eab04be9561961f4725/torchvision-0.20.1-cp311-cp311-win_amd64.whl.metadata
  Using cached torchvision-0.20.1-cp311-cp311-win_amd64.whl.metadata (6.2 kB)
  Obtaining dependency information for ultralytics-thop>=2.0.0 from https://files.pythonhosted.org/packages/4a/87/bfd5285f27c23eeec0f609e814a06fc6c7389f37d8703d98e646a9c5fdc3/ultralytics_thop-2.0.13-py3-none-any.whl.metadata
  Using cached ultralytics_thop-2.0.13-py3-none-any.whl.metadata (9.4 kB)
Using cached ultralytics-8.3.59-py3-none-any.whl (906 kB)
Using cached torchvision-0.20.1-cp311-cp311-win_amd64.whl (1.6 MB

In [1]:
import numpy as np
import cv2
import tensorflow as tf
from ultralytics import YOLO  # YOLOv5 for object detection

# Load the classification model
model = tf.keras.models.load_model('my_model.keras')

# Custom categories
category_dict = {0: 'Biological', 1: 'Cardboard', 2: 'Glass', 3: 'Metal', 4: 'Paper', 5: 'Plastic'}

# Load YOLOv5 pre-trained weights for object detection
yolo_model = YOLO('yolov5s.pt') 

# Start capturing video from the webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Use YOLOv5 to detect objects
    results = yolo_model.predict(frame)  

    # Parse YOLOv5 detection results
    for result in results:  # Loop through detections
        for box in result.boxes:  # Each detection has 'boxes'
            # Extract box coordinates, confidence, and class
            x_min, y_min, x_max, y_max = map(int, box.xyxy[0])  # Convert each coordinate to an integer
            confidence = box.conf[0].item()  # Confidence score
            class_id = int(box.cls[0])  # Class ID
            label = yolo_model.names[class_id]  # YOLO class name

            if confidence > 0.4: 
                # Crop the detected object from the frame
                object_crop = frame[y_min:y_max, x_min:x_max]

                # Preprocess the cropped image for the classification model
                try:
                    object_crop_resized = cv2.resize(object_crop, (100, 100))  # Resize to model input size
                    object_crop_normalized = object_crop_resized / 255.0  # Normalize pixel values
                    object_crop_input = np.expand_dims(object_crop_normalized, axis=0)  # Add batch dimension

                    # Predict the category using the custom classification model
                    predictions = model.predict(object_crop_input)
                    predicted_class = np.argmax(predictions)  # Get the index of the highest probability
                    custom_label = category_dict[predicted_class]  # Map index to category

                    # Draw bounding boxes and custom labels
                    cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
                    cv2.putText(
                        frame,
                        f"{custom_label} ({confidence:.2f})",
                        (x_min, y_min - 10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.5,
                        (255, 255, 255),
                        2,
                    )
                except Exception as e:
                    print(f"Error processing object crop: {e}")

    # Show the processed video frame
    cv2.imshow('Object Detection and Classification', cv2.resize(frame, (1600, 960), interpolation=cv2.INTER_CUBIC))

    # Exit the loop when 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the webcam and close OpenCV windows
cap.release()
cv2.destroyAllWindows()





PRO TIP  Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.


0: 480x640 (no detections), 1845.2ms
Speed: 298.8ms preprocess, 1845.2ms inference, 75.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 398.8ms
Speed: 10.0ms preprocess, 398.8ms inference, 331.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 368.6ms
Speed: 6.0ms preprocess, 368.6ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 335.3ms
Speed: 4.0ms preprocess, 335.3ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 333.6ms
Speed: 4.0ms preprocess, 333.6ms inference, 2.5ms postprocess per image at shape (1, 3, 480, 640)
1/1 [==============================] - 3s 3s/step

0: 480x640 2 persons,

1/1 [==============================] - 0s 30ms/step

0: 480x640 2 cats, 165.1ms
Speed: 3.0ms preprocess, 165.1ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 cat, 178.5ms
Speed: 4.0ms preprocess, 178.5ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)
1/1 [==============================] - 0s 33ms/step

0: 480x640 1 person, 1 cat, 184.9ms
Speed: 2.0ms preprocess, 184.9ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)
1/1 [==============================] - 0s 30ms/step

0: 480x640 1 person, 158.1ms
Speed: 2.0ms preprocess, 158.1ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 166.8ms
Speed: 2.0ms preprocess, 166.8ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 157.6ms
Speed: 4.2ms preprocess, 157.6ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 177.2ms
Speed: 2.0ms 

Speed: 3.0ms preprocess, 191.7ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 196.4ms
Speed: 2.0ms preprocess, 196.4ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 199.2ms
Speed: 3.1ms preprocess, 199.2ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 204.4ms
Speed: 3.1ms preprocess, 204.4ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 193.3ms
Speed: 2.0ms preprocess, 193.3ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 194.6ms
Speed: 2.0ms preprocess, 194.6ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 193.5ms
Speed: 2.0ms preprocess, 193.5ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 192.1ms
Speed: 3.0ms preprocess, 192.1ms inference, 1.0ms postp


0: 480x640 (no detections), 191.6ms
Speed: 3.0ms preprocess, 191.6ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 tv, 193.3ms
Speed: 2.0ms preprocess, 193.3ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
1/1 [==============================] - 0s 31ms/step

0: 480x640 1 person, 1 tv, 195.4ms
Speed: 2.0ms preprocess, 195.4ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
1/1 [==============================] - 0s 34ms/step

0: 480x640 1 tv, 193.1ms
Speed: 3.0ms preprocess, 193.1ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
1/1 [==============================] - 0s 35ms/step

0: 480x640 1 person, 1 tv, 1 laptop, 189.0ms
Speed: 2.0ms preprocess, 189.0ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
1/1 [==============================] - 0s 36ms/step

0: 480x640 1 tv, 1 laptop, 205.0ms
Speed: 4.0ms preprocess, 205.0ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 64

Speed: 2.0ms preprocess, 208.8ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 tv, 190.1ms
Speed: 2.0ms preprocess, 190.1ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 tv, 194.1ms
Speed: 4.0ms preprocess, 194.1ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 189.1ms
Speed: 2.0ms preprocess, 189.1ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 194.1ms
Speed: 3.0ms preprocess, 194.1ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 193.1ms
Speed: 2.0ms preprocess, 193.1ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 191.2ms
Speed: 4.0ms preprocess, 191.2ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 192.1ms
Speed: 3.0ms preprocess, 192.1ms inference, 2.0ms postprocess per image at sh


0: 480x640 1 person, 259.1ms
Speed: 2.2ms preprocess, 259.1ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
1/1 [==============================] - 0s 34ms/step

0: 480x640 1 person, 182.6ms
Speed: 3.0ms preprocess, 182.6ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)
1/1 [==============================] - 0s 37ms/step

0: 480x640 1 person, 193.5ms
Speed: 2.1ms preprocess, 193.5ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
1/1 [==============================] - 0s 30ms/step

0: 480x640 1 person, 169.7ms
Speed: 2.0ms preprocess, 169.7ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
1/1 [==============================] - 0s 28ms/step

0: 480x640 1 person, 171.1ms
Speed: 5.0ms preprocess, 171.1ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)
1/1 [==============================] - 0s 34ms/step

0: 480x640 1 person, 189.5ms
Speed: 2.1ms preprocess, 189.5ms inference, 1.5ms postprocess per imag